In [1]:
import pandas as pd

In [2]:
# Load Excel dataset and confirm row and column counts; sheet name preserved exactly from Numbers to Excel conversion
athletes_raw = pd.read_excel(
    'olympics_dataset 5.xlsx',
    sheet_name='Sheet 1 - olympics_dataset 5'
)

athletes_raw.shape

(252565, 11)

In [3]:
# Confirm Medal uses 'no medal' string vs. NaN (affects calculated fields later)
athletes_raw['Medal'].value_counts()

Medal
No medal    213747
Bronze       13070
Gold         13002
Silver       12746
Name: count, dtype: int64

In [4]:
# 1906 year not IOC-recognized; keeps scope to official Summer Games
athletes_raw = athletes_raw[athletes_raw['Year'] != 1906]
athletes_raw.shape

(250832, 11)

In [5]:
# Exact duplicates on this key concentrate in Art Competitions (1912 - 1948 data entry errors)
athletes_raw = athletes_raw.drop_duplicates(subset = ['Name', 'NOC', 'Year', 'Event', 'Medal'])
athletes_raw.shape

(249360, 11)

In [6]:
# Player id overlaps across source datasets and cannot serve as a reliable athlete key
athletes_raw = athletes_raw.drop(columns = ['player_id'])
athletes_raw.shape

(249360, 10)

In [7]:
# Historical NOC codes will not resolve on a modern map without recoding to current country names
historical_teams = {
    'Soviet Union': 'Russia',
    'Unified Team': 'Russia',
    'East Germany': 'Germany',
    'West Germany': 'Germany',
    'Unified German Team': 'Germany',
    'Yugoslavia': 'Serbia',
    'Czechoslovakia': 'Czechia',
    'Australasia': 'Australia',
}
exclude_map = {'Bohemia', 'Independent Olympic Athletes',
               'Refugee Olympic Team', 'Mixed team'}

def get_country(team):
    if team in exclude_map:
        return 'Mixed/Historical Team'
    return historical_teams.get(team, team)

athletes_clean = athletes_raw.copy()
athletes_clean['Country_for_Map'] = athletes_clean['Team'].apply(get_country)
athletes_clean['Country_for_Map'].value_counts().head(10)

Country_for_Map
United States    15902
Germany          13015
Great Britain    11145
France           11040
Russia            9319
Italy             9089
Australia         8329
Canada            7678
Japan             7598
Hungary           6427
Name: count, dtype: int64

In [8]:
# Numbers to Excel conversion corrupted these three names; identity confirmed by unique medal counts
name_corrections = {
    'Michael Ii': 'Michael Phelps',
    'Larysa (diriy-)': 'Larisa Latynina',
    'Natalie (-hall)': 'Natalie Coughlin',
}
athletes_clean['Name'] = athletes_clean['Name'].replace(name_corrections)
athletes_clean[athletes_clean['Medal'] != 'No medal'].groupby('Name').size().sort_values(ascending=False).head(10)

Name
Michael Phelps         28
Larisa Latynina        18
Nikolay Andrianov      15
Charles Jr.            15
Borys Shakhlin         13
Takashi Ono            13
Edoardo Mangiarotti    13
John Jr.               13
Ryan Lochte            12
Aleksey Nemov          12
dtype: int64

In [9]:
# Tableau connects to CSV; avoids xlsx dependency outside the cleaning notebook
athletes_clean.to_csv('olympics_clean.csv', index=False)